# Toward Context-Aware Belief-State Modeling for Prediction Markets

This notebook is the current empirical foundation for a NeurIPS paper built around prediction markets as strategic probabilistic sensors rather than next-tick trading targets.

The central question is not whether a generic model can beat Polymarket everywhere. The stronger question is:

**What does the market already know, when should it be trusted, when is it about to revise its beliefs, and what kind of context should a future encoder retrieve and compress?**

## Paper Spine

The notebook is organized around four claims that can support a serious benchmark-and-analysis paper:

1. **Terminal forecasting is already hard to improve on average** because raw market price is a strong baseline.
2. **Trust is partly simple but still scientifically meaningful**: confidence margin already explains a lot about whether a state is reliable.
3. **Large repricing is predictable from market-native state**, while external crypto shocks appear weaker and more selective than naive stories suggest.
4. **Text and tag similarity are useful for context retrieval**, which motivates future hierarchical encoder designs more than raw global fusion does.


## Environment and Imports

This cell imports the analysis stack and the existing benchmark helpers. Reusing the repository utilities keeps the paper notebook aligned with the benchmark package rather than creating a disconnected exploratory pipeline.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from polymarket_research import PolymarketDataset
from polymarket_research.utils import setup_root

REPO_ROOT = setup_root()

from benchmarks.benchmark_utils import (
    build_repricing_dataset,
    load_snapshot_frame,
    make_feature_matrix,
    make_rolling_splits,
    prepare_resolved_markets,
)
from benchmarks.covariate_utils import load_covariate_config, merge_covariates_asof


## Configuration and Shared Helpers

The notebook excludes the Polymarket `crypto` domain because it is incomplete and noisy for the current paper draft. We still use `BTC/USD` and `ETH/USD` as **external** liquid signals.

A key design choice is to keep the context features weakly supervised and interpretable:

- `family_id` is a heuristic grouping from domain, tags, and normalized question text
- related-market context is built from contemporaneous sibling snapshots
- external shocks are defined through rolling return z-scores rather than an opaque event detector

That makes the notebook scientifically honest: the goal is to discover what kinds of structure are promising before investing in a full learned encoder.

In [2]:
DB_PATH = DEFAULT_DB_PATH
DOMAINS = ('politics', 'geopolitics', 'technology', 'finance_economy')
MAX_MARKETS_PER_DOMAIN = 120
MIN_PROBABILITY_ROWS = 288

TERMINAL_HORIZONS = (24, 72, 168)
REPRICING_FUTURE_HOURS = 24
REPRICING_LOOKBACK_HOURS = 24
REPRICING_SAMPLE_EVERY_HOURS = 12
REPRICING_MOVE_THRESHOLD = 0.15

SHOCK_Z_THRESHOLD = 2.0
SHOCK_STD_WINDOW = 288
EXTERNAL_PATH = REPO_ROOT / 'cached_data' / 'external_covariates'

CLOSENESS_MAX_MARKETS_PER_DOMAIN = 70
TOP_K = 5

def parse_listish(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except Exception:
        pass
    if '|' in text:
        parts = text.split('|')
    elif ',' in text:
        parts = text.split(',')
    else:
        parts = [text]
    return [part.strip() for part in parts if part.strip()]

def normalize_text(value: str) -> str:
    text = str(value or '').lower()
    text = re.sub(r'[^a-z0-9\s]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def build_family_id(question: str, domain: str, tags) -> str:
    norm_q = normalize_text(question)
    norm_tags = [normalize_text(tag) for tag in parse_listish(tags)]
    tokens = [tok for tok in norm_q.split() if tok not in {'will', 'the', 'a', 'an', 'be', 'is', 'are', 'to', 'of', 'by', 'in'}]
    key = ' '.join(tokens[:6]) if tokens else norm_q[:48]
    tag_key = '|'.join(sorted(norm_tags[:3]))
    return f"{domain}::{tag_key}::{key}".strip(':')

def latest_context_snapshot(probabilities_df: pd.DataFrame, market_id: str, cutoff: pd.Timestamp):
    panel = probabilities_df.loc[(probabilities_df['market_id'] == market_id) & (probabilities_df['timestamp_utc'] <= cutoff)]
    if panel.empty:
        return None
    return panel.iloc[-1]

def build_family_context_features(dataset: pd.DataFrame, probabilities_df: pd.DataFrame, market_meta: pd.DataFrame) -> pd.DataFrame:
    family_map = market_meta.groupby('family_id')['market_id'].apply(list).to_dict()
    rows = []
    for row in dataset[['market_id', 'cutoff_timestamp_utc', 'market_price_baseline']].itertuples(index=False):
        family_id = market_meta.loc[market_meta['market_id'] == row.market_id, 'family_id'].iloc[0]
        related_ids = [mid for mid in family_map.get(family_id, []) if mid != row.market_id]
        related_probs = []
        related_trade_flags = []
        for related_id in related_ids:
            snap = latest_context_snapshot(probabilities_df, related_id, row.cutoff_timestamp_utc)
            if snap is None:
                continue
            related_probs.append(float(snap['yes_probability']))
            related_trade_flags.append(float(snap['observed_trade']))
        rows.append({
            'market_id': row.market_id,
            'cutoff_timestamp_utc': row.cutoff_timestamp_utc,
            'family_related_count': float(len(related_probs)),
            'family_prob_mean': float(np.mean(related_probs)) if related_probs else np.nan,
            'family_prob_std': float(np.std(related_probs)) if related_probs else np.nan,
            'family_prob_gap': float(np.max(related_probs) - np.min(related_probs)) if related_probs else np.nan,
            'family_trade_share_mean': float(np.mean(related_trade_flags)) if related_trade_flags else np.nan,
            'family_vs_market_gap': float(abs(np.mean(related_probs) - row.market_price_baseline)) if related_probs else np.nan,
        })
    return pd.DataFrame(rows)

def build_shock_table(path: Path, z_threshold: float = 2.0, std_window: int = 288) -> pd.DataFrame:
    covariates = load_external_covariates(path)
    wide = pivot_covariates_to_wide(covariates, value_col='value').sort_values('timestamp_utc').reset_index(drop=True)
    out = wide[['timestamp_utc']].copy()
    value_cols = [col for col in wide.columns if col != 'timestamp_utc']
    for col in value_cols:
        series = pd.to_numeric(wide[col], errors='coerce')
        ret = series.pct_change()
        sigma = ret.rolling(std_window, min_periods=max(24, std_window // 6)).std()
        z = ret / sigma.replace(0.0, np.nan)
        out[f'{col}_level'] = series
        out[f'{col}_ret'] = ret
        out[f'{col}_z'] = z
        out[f'{col}_shock'] = (z.abs() >= z_threshold).astype(float)
    out['any_external_shock'] = out[[col for col in out.columns if col.endswith('_shock')]].max(axis=1)
    return out

def tag_jaccard(tags_a, tags_b):
    a = set(parse_listish(tags_a))
    b = set(parse_listish(tags_b))
    if not a and not b:
        return 0.0
    return len(a & b) / len(a | b)

def fit_pca_on_train(train_df: pd.DataFrame, test_df: pd.DataFrame, cols, n_components: int = 6, prefix: str = 'z'):
    cols = [col for col in cols if col in train_df.columns]
    if not cols:
        return pd.DataFrame(index=train_df.index), pd.DataFrame(index=test_df.index)
    n_components = max(1, min(int(n_components), len(cols), max(1, len(train_df) - 1)))
    imputer = SimpleImputer(strategy='median')
    scaler = StandardScaler()
    train_x = scaler.fit_transform(imputer.fit_transform(train_df[cols]))
    test_x = scaler.transform(imputer.transform(test_df[cols]))
    pca = PCA(n_components=n_components, random_state=0)
    train_z = pca.fit_transform(train_x)
    test_z = pca.transform(test_x)
    train_out = pd.DataFrame(train_z, index=train_df.index, columns=[f'{prefix}_{i+1}' for i in range(train_z.shape[1])])
    test_out = pd.DataFrame(test_z, index=test_df.index, columns=[f'{prefix}_{i+1}' for i in range(test_z.shape[1])])
    return train_out, test_out

def clipped(p):
    return np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)

def safe_auc(y_true, score):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, score)

def safe_ap(y_true, score):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return average_precision_score(y_true, score)

def recall_at_k(pair_df: pd.DataFrame, score_col: str, label_col: str, k: int = 5) -> float:
    recalls = []
    for market_id, group in pair_df.groupby('market_id_a', sort=False):
        positives = int(group[label_col].sum())
        if positives == 0:
            continue
        top = group.sort_values(score_col, ascending=False).head(k)
        recalls.append(float(top[label_col].sum() > 0))
    return float(np.mean(recalls)) if recalls else np.nan


## Load Markets, Probability Histories, and External Signals

This is the main data-construction step. We load resolved markets from four domains, build terminal and repricing panels, construct related-market context summaries, and attach external `BTC/USD` and `ETH/USD` signals.

The result is not yet a learned representation; it is a clean empirical scaffold for testing what kinds of context matter and where simple baselines already dominate.

In [3]:
DATASET_ARTEFACT_DIR = REPO_ROOT / 'research_notebooks' / 'running_artefacts'

dataset = PolymarketDataset.from_parquet(DATASET_ARTEFACT_DIR)
markets = dataset.markets.copy()
probabilities = dataset.probabilities.copy()

prepared_markets = prepare_resolved_markets(markets)
prepared_markets = prepared_markets[prepared_markets['domain'].isin(DOMAINS)].copy()


Loaded terminal rows: 966 from 342 markets
Loaded repricing rows: 80755 from 453 markets


## Coverage Overview

Before evaluating any model, it is important to know what the notebook actually covers. These summaries establish the sample sizes by domain and horizon and give a quick view of how frequent large repricing events are.

This also helps calibrate how ambitious each claim can be: terminal forecasting here is much smaller than repricing, so it should be read as a reference task rather than a brute-force data regime.

In [4]:
terminal_summary = (
    terminal.groupby(['primary_domain', 'horizon_name'], dropna=False)
    .agg(rows=('market_id', 'size'), markets=('market_id', 'nunique'), positive_rate=('target', 'mean'))
    .reset_index()
)
repricing_summary = (
    repricing.groupby('primary_domain', dropna=False)
    .agg(rows=('market_id', 'size'), markets=('market_id', 'nunique'), repricing_rate=('target', 'mean'))
    .reset_index()
)
display(terminal_summary)
display(repricing_summary)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.barplot(data=terminal_summary, x='rows', y='primary_domain', hue='horizon_name', ax=axes[0], palette='crest')
axes[0].set_title('Terminal panel coverage')
axes[0].set_xlabel('Rows')
axes[0].set_ylabel('')

sns.barplot(data=repricing_summary, x='rows', y='primary_domain', ax=axes[1], palette='flare')
axes[1].set_title('Repricing panel coverage')
axes[1].set_xlabel('Rows')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

## 1. Terminal Forecasting as a Reference Task

This section is intentionally modest. We compare `market_price` against a few simple feature-based baselines to establish a core empirical fact: **average terminal forecasting is already dominated by the current market probability**.

That is not a failure. It is the benchmark condition that makes the problem scientifically interesting. If the market is already strong, a future model should not be judged by naive average uplift alone; it should instead focus on hard states, trust, belief revision, or selective improvements.

In [5]:
local_a_cols = [
    'current_yes_probability', 'confidence_margin', 'snapshot_staleness_hours', 'observed_trade_now', 'trade_count_now', 'total_size_now', 'last_trade_price_now',
    'lookback_24h_yes_probability_change', 'lookback_24h_volatility', 'lookback_24h_abs_move_mean', 'lookback_24h_abs_move_max',
    'lookback_168h_yes_probability_change', 'lookback_168h_volatility', 'lookback_168h_abs_move_mean', 'lookback_168h_abs_move_max',
    'hours_to_resolution', 'market_age_hours', 'life_progress', 'horizon_hours',
] + [col for col in terminal.columns if col.startswith('domain_')]

context_cols = ['family_related_count', 'family_prob_mean', 'family_prob_std', 'family_prob_gap', 'family_trade_share_mean', 'family_vs_market_gap']
external_cols = [col for col in terminal.columns if col.startswith('btc_usd_') or col.startswith('eth_usd_')]

terminal_variants = {
    'market_price': None,
    'raw(A)': {'base': local_a_cols, 'encode': []},
    'raw(A)+raw(B)': {'base': local_a_cols + context_cols, 'encode': []},
    'raw(A)+raw(B)+raw(E)': {'base': local_a_cols + context_cols + external_cols, 'encode': []},
    'raw(A)+encoded(B,E)': {'base': local_a_cols, 'encode': context_cols + external_cols},
}

terminal_rows = []
for train_df, test_df, meta in rolling_time_splits(terminal, time_col='end_date', n_splits=4, min_train_fraction=0.5):
    y_test = test_df['target'].astype(int).to_numpy()
    base_prob = clipped(test_df['market_price_baseline'])
    terminal_rows.append({
        'variant': 'market_price',
        'fold': meta['fold'],
        'log_loss': log_loss(y_test, base_prob),
        'brier': brier_score_loss(y_test, base_prob),
        'roc_auc': safe_auc(y_test, base_prob),
    })
    for variant_name, spec in terminal_variants.items():
        if variant_name == 'market_price':
            continue
        train_parts = [train_df[[col for col in spec['base'] if col in train_df.columns]].copy()]
        test_parts = [test_df[[col for col in spec['base'] if col in test_df.columns]].copy()]
        if spec['encode']:
            z_train, z_test = fit_pca_on_train(train_df, test_df, spec['encode'], n_components=6, prefix='z')
            train_parts.append(z_train)
            test_parts.append(z_test)
        x_train = pd.concat(train_parts, axis=1)
        x_test = pd.concat(test_parts, axis=1)
        model = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(max_iter=2000)),
        ])
        model.fit(x_train, train_df['target'].astype(int).to_numpy())
        p_test = clipped(model.predict_proba(x_test)[:, 1])
        terminal_rows.append({
            'variant': variant_name,
            'fold': meta['fold'],
            'log_loss': log_loss(y_test, p_test),
            'brier': brier_score_loss(y_test, p_test),
            'roc_auc': safe_auc(y_test, p_test),
        })

terminal_metrics = pd.DataFrame(terminal_rows)
terminal_summary_table = terminal_metrics.groupby('variant', dropna=False)[['log_loss', 'brier', 'roc_auc']].mean().sort_values('log_loss').reset_index()
display(terminal_summary_table)

The expected reading of the next plot is not “which model wins the leaderboard?” but rather “how dominant is the raw market price baseline, and how far are naive context additions from challenging it?”

If the gap remains large, that is evidence against generic tabular fusion and in favor of a more structured future model.

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=terminal_summary_table, x='log_loss', y='variant', ax=axes[0], palette='crest')
axes[0].set_title('Terminal forecasting: mean log loss')
axes[0].set_xlabel('Lower is better')
axes[0].set_ylabel('')

sns.barplot(data=terminal_summary_table.sort_values('roc_auc', ascending=False), x='roc_auc', y='variant', ax=axes[1], palette='flare')
axes[1].set_title('Terminal forecasting: mean ROC-AUC')
axes[1].set_xlabel('Higher is better')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

## 2. Trustworthiness as a Meta-Prediction Task

Instead of asking only what the event probability is, we can ask whether the current market state is reliable at all.

For a clean first probe, we define a `bad_state` as a terminal snapshot with large absolute terminal error (`market_abs_error >= 0.25`). This is a deliberately simple trust target. The point is not that it is the only trust definition, but that it gives a measurable test of whether reliability can be predicted from current market state.

In [7]:
trust_df = terminal.copy()
trust_df['bad_state'] = (trust_df['market_abs_error'] >= 0.25).astype(int)
trust_features = local_a_cols + context_cols

trust_rows = []
coverage_rows = []
coverage_grid = np.linspace(0.1, 1.0, 10)

for train_df, test_df, meta in rolling_time_splits(trust_df, time_col='end_date', n_splits=4, min_train_fraction=0.5):
    y_test = test_df['bad_state'].astype(int).to_numpy()
    baseline_risk = 1.0 - 2.0 * test_df['confidence_margin'].to_numpy()
    trust_rows.append({
        'variant': 'confidence_margin',
        'fold': meta['fold'],
        'ap': safe_ap(y_test, baseline_risk),
        'roc_auc': safe_auc(y_test, baseline_risk),
    })
    model = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000, class_weight='balanced')),
    ])
    model.fit(train_df[trust_features], train_df['bad_state'].astype(int).to_numpy())
    learned_risk = model.predict_proba(test_df[trust_features])[:, 1]
    trust_rows.append({
        'variant': 'learned_trust',
        'fold': meta['fold'],
        'ap': safe_ap(y_test, learned_risk),
        'roc_auc': safe_auc(y_test, learned_risk),
    })
    for variant_name, risk_score in [('confidence_margin', baseline_risk), ('learned_trust', learned_risk)]:
        order = np.argsort(risk_score)
        sorted_df = test_df.iloc[order].copy()
        for coverage in coverage_grid:
            keep_n = max(1, int(len(sorted_df) * coverage))
            kept = sorted_df.iloc[:keep_n]
            coverage_rows.append({
                'variant': variant_name,
                'fold': meta['fold'],
                'coverage': coverage,
                'bad_state_rate': kept['bad_state'].mean(),
                'mean_abs_error': kept['market_abs_error'].mean(),
            })

trust_metrics = pd.DataFrame(trust_rows)
trust_curve = pd.DataFrame(coverage_rows)
display(trust_metrics.groupby('variant', dropna=False)[['ap', 'roc_auc']].mean().reset_index())

The next figure shows the selective-prediction view of trust. Lower retained error at low coverage means the scoring rule is better at identifying market states that are genuinely safer to trust.

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
curve_summary = trust_curve.groupby(['variant', 'coverage'], dropna=False)[['bad_state_rate', 'mean_abs_error']].mean().reset_index()
sns.lineplot(data=curve_summary, x='coverage', y='bad_state_rate', hue='variant', marker='o', ax=axes[0])
axes[0].set_title('Selective trust: retained bad-state rate')
axes[0].set_xlabel('Coverage kept')
axes[0].set_ylabel('Lower is better')

sns.lineplot(data=curve_summary, x='coverage', y='mean_abs_error', hue='variant', marker='o', ax=axes[1])
axes[1].set_title('Selective trust: retained mean absolute error')
axes[1].set_xlabel('Coverage kept')
axes[1].set_ylabel('Lower is better')
plt.tight_layout()
plt.show()

## 3. Repricing and External Shock Propagation

Large repricing is the strongest dynamic task in the current setup because it can be sampled repeatedly along the life of a market.

This section asks two related questions:

1. Are large future revisions predictable from market-native state?
2. Do external `BTC/USD` and `ETH/USD` shocks add meaningful explanatory power, or is their role weaker and more selective than naive narratives suggest?

In [9]:
base_cols = [
    'current_yes_probability', 'confidence_margin', 'hours_to_resolution', 'life_progress',
    'recent_abs_move_mean', 'recent_abs_move_max', 'recent_volatility', 'recent_directional_move',
    'observed_trade_share', 'trade_count_sum', 'total_size_sum',
] + [col for col in repricing.columns if col.startswith('domain_')]

shock_cols = ['btc_usd_ret', 'btc_usd_z', 'btc_usd_shock', 'eth_usd_ret', 'eth_usd_z', 'eth_usd_shock', 'any_external_shock']

shock_descriptive = (
    repricing.groupby('btc_or_eth_shock', dropna=False)
    .agg(
        rows=('market_id', 'size'),
        markets=('market_id', 'nunique'),
        repricing_rate=('target', 'mean'),
        mean_abs_future_move=('future_move', lambda s: np.mean(np.abs(s))),
    )
    .reset_index()
)
display(shock_descriptive)

repricing_rows = []
for train_df, test_df, meta in rolling_time_splits(repricing, time_col='timestamp_utc', n_splits=4, min_train_fraction=0.5):
    y_test = test_df['target'].astype(int).to_numpy()
    for variant_name, cols in [
        ('microstructure', base_cols),
        ('microstructure+shocks', base_cols + shock_cols),
        ('shocks_only', shock_cols + [col for col in repricing.columns if col.startswith('domain_')]),
    ]:
        model = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(max_iter=2000, class_weight='balanced')),
        ])
        model.fit(train_df[cols], train_df['target'].astype(int).to_numpy())
        p_test = clipped(model.predict_proba(test_df[cols])[:, 1])
        repricing_rows.append({
            'variant': variant_name,
            'fold': meta['fold'],
            'average_precision': safe_ap(y_test, p_test),
            'roc_auc': safe_auc(y_test, p_test),
            'log_loss': log_loss(y_test, p_test),
        })

repricing_metrics = pd.DataFrame(repricing_rows)
repricing_summary_table = repricing_metrics.groupby('variant', dropna=False)[['average_precision', 'roc_auc', 'log_loss']].mean().sort_values('average_precision', ascending=False).reset_index()
display(repricing_summary_table)

The plots below separate two claims:

- the left panel is a **descriptive event-study view** of shock vs non-shock states by domain;
- the right panel is a **predictive comparison** between market-native and shock-augmented models.

This combination matters because a weak raw shock effect can still coexist with a strong repricing benchmark driven mainly by market microstructure.

In [10]:
domain_response = (
    repricing.groupby(['primary_domain', 'btc_or_eth_shock'], dropna=False)
    .agg(
        rows=('market_id', 'size'),
        repricing_rate=('target', 'mean'),
        mean_abs_future_move=('future_move', lambda s: np.mean(np.abs(s))),
    )
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.barplot(data=domain_response, x='repricing_rate', y='primary_domain', hue='btc_or_eth_shock', ax=axes[0], palette='crest')
axes[0].set_title('Repricing rate by domain and shock state')
axes[0].set_xlabel('Higher is more reactive')
axes[0].set_ylabel('')

sns.barplot(data=repricing_summary_table, x='average_precision', y='variant', ax=axes[1], palette='flare')
axes[1].set_title('Repricing prediction: mean average precision')
axes[1].set_xlabel('Higher is better')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

example_cols = [
    'timestamp_utc', 'primary_domain', 'question', 'current_yes_probability', 'future_move', 'target',
    'btc_usd_z', 'eth_usd_z', 'btc_usd_shock', 'eth_usd_shock'
]
shock_examples = (
    repricing.loc[repricing['btc_or_eth_shock'] >= 1.0, example_cols]
    .assign(abs_future_move=lambda x: x['future_move'].abs())
    .sort_values('abs_future_move', ascending=False)
    .head(12)
)
display(shock_examples)

## 4. Retrieval Signals for Future Context Encoders

The previous sections show that naive context fusion is not enough. This makes retrieval quality itself important: if a future encoder is going to consume related markets `B`, it needs a good candidate set.

This section therefore studies closeness signals directly. The key question is: **which primitive similarity channels are strong enough to recover plausible neighbors under weak labels such as `same_family`?**

In [11]:
retrieval_frames = []
for domain in DOMAINS:
    frame = markets.loc[markets['domain'] == domain].copy()
    if 'probability_rows' in frame.columns:
        frame = frame.loc[frame['probability_rows'].fillna(0) >= MIN_PROBABILITY_ROWS]
    if CLOSENESS_MAX_MARKETS_PER_DOMAIN is not None:
        frame = frame.sort_values('volume_num', ascending=False).head(CLOSENESS_MAX_MARKETS_PER_DOMAIN)
    retrieval_frames.append(frame)

retrieval_markets = pd.concat(retrieval_frames, ignore_index=True)
retrieval_probs = probabilities.loc[probabilities['market_id'].isin(retrieval_markets['market_id'])].copy()

retrieval_markets['family_id'] = [build_family_id(q, d, tags) for q, d, tags in zip(retrieval_markets['question'], retrieval_markets['primary_domain'], retrieval_markets['tag_labels'], strict=False)]
retrieval_markets['text_blob'] = retrieval_markets['question'].fillna('') + ' ' + retrieval_markets['description'].fillna('') + ' ' + retrieval_markets['tag_labels'].fillna('')

vectorizer = TfidfVectorizer(min_df=2, max_features=5000, ngram_range=(1, 2), stop_words='english')
tfidf = vectorizer.fit_transform(retrieval_markets['text_blob'])
text_cos = cosine_similarity(tfidf)

prob_hourly = (
    retrieval_probs[['market_id', 'timestamp_utc', 'yes_probability']]
    .dropna()
    .assign(timestamp_utc=lambda df: pd.to_datetime(df['timestamp_utc']).dt.floor('1h'))
    .groupby(['market_id', 'timestamp_utc'], as_index=False)['yes_probability']
    .mean()
)


A future hierarchical encoder only needs a **good enough** retrieval layer, not a perfect oracle. The next plot therefore emphasizes relative ranking between similarity channels rather than absolute perfection.

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.barplot(data=retrieval_metrics, x='roc_auc', y='score', hue='label', ax=axes[0], palette='crest')
axes[0].set_title('Retrieval quality under weak labels: ROC-AUC')
axes[0].set_xlabel('Higher is better')
axes[0].set_ylabel('')

sns.barplot(data=retrieval_metrics, x='recall_at_5', y='score', hue='label', ax=axes[1], palette='flare')
axes[1].set_title('Retrieval quality under weak labels: recall@5')
axes[1].set_xlabel('Higher is better')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

## Main Takeaways

This notebook supports the following paper-ready narrative:

- **Raw market price is a very strong terminal baseline.** Average terminal uplift is therefore the wrong headline objective.
- **Trust is a real object, but simple market-native confidence is already powerful.** This argues for careful selective-prediction framing rather than overclaiming learned trust gains.
- **Large repricing is the strongest dynamic prediction task.** Market-native microstructure already contains substantial signal, while naive external shock conditioning adds little on average.
- **External shocks are not useless, but they are selective rather than universal.** That makes them better as supporting context than as a standalone predictive story.
- **Text and tag similarity provide a strong retrieval prior for related markets.** This is the cleanest empirical support for a future hierarchical encoder with `target market + retrieved context` rather than a flat fusion model.

In other words, the notebook does not yet prove that a compact latent encoder wins. It proves something almost as important for a serious paper: **where the easy stories fail, where the strong baselines already are, and which structural ingredients are worth encoding next.**